# NB06 — Peer Group Analysis & Benchmarking

This notebook compares each hospital to peers of similar size, teaching status, and ownership. Hospitals whose CMI or severity mix falls significantly below their peer group average may have documentation gaps.

**Key Concept**: The peer_group column (created in NB01) combines three dimensions:
- **Bed Size Tier**: Hospital capacity category (e.g., "200-399" beds)
- **Teaching Status**: Whether the hospital is affiliated with medical education ("teaching" or "non_teaching")
- **Ownership Category**: Hospital ownership type (e.g., "nonprofit", "for_profit", "public")

This creates peer groups like `"200-399_teaching_nonprofit"`. We expect ~49 unique groups across the dataset.

**Documentation Gap Hypothesis**: If a hospital's CMI or MCC ratio is significantly below its peer group's average, it may indicate incomplete documentation of patient complexity, leading to undercoding and potential revenue loss.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup paths
input_file = Path('../../data/outputs/nb05_features/hospital_features.csv')
output_dir = Path('../../data/outputs/nb06_peer_benchmarks')
output_dir.mkdir(parents=True, exist_ok=True)

# Load feature-enriched dataset
df = pd.read_csv(input_file, dtype={'ccn': str})

print(f"Dataset shape: {df.shape}")
print(f"\nDataset info:")
print(df.info())
print(f"\nFirst few rows:")
print(df.head())

# Check peer groups
n_peer_groups = df['peer_group'].nunique()
print(f"\nNumber of peer groups: {n_peer_groups}")
print(f"\nPeer group distribution:")
group_sizes = df['peer_group'].value_counts()
print(group_sizes)
print(f"\nGroup size statistics:")
print(f"  Mean hospitals per group: {group_sizes.mean():.1f}")
print(f"  Median hospitals per group: {group_sizes.median():.1f}")
print(f"  Min hospitals per group: {group_sizes.min()}")
print(f"  Max hospitals per group: {group_sizes.max()}")

Dataset shape: (3280, 50)

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3280 entries, 0 to 3279
Data columns (total 50 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ccn                         3280 non-null   object 
 1   hospital_name               3280 non-null   object 
 2   city                        3280 non-null   object 
 3   state                       3280 non-null   object 
 4   zip_code                    3280 non-null   int64  
 5   county                      3280 non-null   object 
 6   beds                        3274 non-null   float64
 7   bed_size_tier               3274 non-null   object 
 8   ownership                   3280 non-null   object 
 9   ownership_category          3280 non-null   object 
 10  is_teaching                 3280 non-null   bool   
 11  is_urban                    3280 non-null   bool   
 12  census_region               3218 non-null   objec

## Peer Group Statistics

We compute aggregate statistics for each peer group across key metrics: CMI (Case Mix Index), MCC ratio (Major Complication/Comorbidity rate), severity index, average payment per discharge, and bed size. These serve as the baseline for identifying hospitals with potential documentation gaps.

In [2]:
# Compute peer group aggregates
peer_stats = df.groupby('peer_group').agg(
    n_hospitals=('ccn', 'count'),
    peer_cmi_mean=('cmi', 'mean'),
    peer_cmi_median=('cmi', 'median'),
    peer_cmi_std=('cmi', 'std'),
    peer_cmi_p25=('cmi', lambda x: x.quantile(0.25)),
    peer_cmi_p75=('cmi', lambda x: x.quantile(0.75)),
    peer_mcc_ratio_mean=('mcc_ratio', 'mean'),
    peer_severity_index_mean=('severity_index', 'mean'),
    peer_avg_payment=('avg_payment_per_discharge', 'mean'),
    peer_beds_mean=('beds', 'mean'),
).reset_index()

print(f"Peer group statistics before filtering:")
print(f"  Total groups: {len(peer_stats)}")
print(f"\n{peer_stats.head(10)}")

# Filter to groups with at least 5 hospitals
min_group_size = 5
peer_stats_filtered = peer_stats[peer_stats['n_hospitals'] >= min_group_size].copy()
dropped_groups = len(peer_stats) - len(peer_stats_filtered)

print(f"\nFiltering to groups with >= {min_group_size} hospitals:")
print(f"  Groups kept: {len(peer_stats_filtered)}")
print(f"  Groups dropped (too small): {dropped_groups}")

if dropped_groups > 0:
    small_groups = peer_stats[peer_stats['n_hospitals'] < min_group_size]
    print(f"\n  Dropped groups (n < {min_group_size}):")
    print(small_groups[['peer_group', 'n_hospitals']])

Peer group statistics before filtering:
  Total groups: 49

                        peer_group  n_hospitals  peer_cmi_mean  \
0     1-24_non_teaching_for_profit           53       1.982526   
1     1-24_non_teaching_government           13       1.081540   
2      1-24_non_teaching_nonprofit           33       1.706477   
3          1-24_non_teaching_other           35       2.572087   
4         1-24_teaching_for_profit            5       2.080180   
5         1-24_teaching_government            3       1.077200   
6          1-24_teaching_nonprofit            4       1.690950   
7              1-24_teaching_other            6       2.754667   
8  100-199_non_teaching_for_profit          141       1.616234   
9  100-199_non_teaching_government           59       1.542512   

   peer_cmi_median  peer_cmi_std  peer_cmi_p25  peer_cmi_p75  \
0          1.99865      0.775253      1.247450      2.475950   
1          1.08530      0.156105      0.971325      1.197600   
2          1.33100   

In [3]:
# Merge peer stats back to hospital-level data
df_benchmarked = df.merge(
    peer_stats_filtered[[
        'peer_group', 'n_hospitals', 'peer_cmi_mean', 'peer_cmi_median', 
        'peer_cmi_std', 'peer_cmi_p25', 'peer_cmi_p75',
        'peer_mcc_ratio_mean', 'peer_severity_index_mean', 
        'peer_avg_payment', 'peer_beds_mean'
    ]],
    on='peer_group',
    how='left'
)

# Compute deviation metrics
df_benchmarked['cmi_gap'] = df_benchmarked['cmi'] - df_benchmarked['peer_cmi_mean']

# Handle z-score calculation with NaN std (groups with 1 hospital)
df_benchmarked['cmi_z_score'] = 0.0
valid_std = (df_benchmarked['peer_cmi_std'].notna()) & (df_benchmarked['peer_cmi_std'] > 0)
df_benchmarked.loc[valid_std, 'cmi_z_score'] = (
    (df_benchmarked.loc[valid_std, 'cmi'] - df_benchmarked.loc[valid_std, 'peer_cmi_mean']) / 
    df_benchmarked.loc[valid_std, 'peer_cmi_std']
)

df_benchmarked['mcc_gap'] = df_benchmarked['mcc_ratio'] - df_benchmarked['peer_mcc_ratio_mean']
df_benchmarked['severity_gap'] = df_benchmarked['severity_index'] - df_benchmarked['peer_severity_index_mean']
df_benchmarked['payment_gap'] = df_benchmarked['avg_payment_per_discharge'] - df_benchmarked['peer_avg_payment']

print(f"Data after merging peer statistics:")
print(f"  Shape: {df_benchmarked.shape}")
print(f"  Rows with peer group stats: {df_benchmarked['n_hospitals'].notna().sum()}")
print(f"  Rows without peer group stats (dropped groups): {df_benchmarked['n_hospitals'].isna().sum()}")

print(f"\nCMI Gap Distribution:")
print(df_benchmarked['cmi_gap'].describe())
print(f"\nMCC Gap Distribution:")
print(df_benchmarked['mcc_gap'].describe())
print(f"\nCMI Z-Score Distribution:")
print(df_benchmarked['cmi_z_score'].describe())

Data after merging peer statistics:
  Shape: (3280, 65)
  Rows with peer group stats: 3266
  Rows without peer group stats (dropped groups): 14

CMI Gap Distribution:
count    3.047000e+03
mean    -4.547287e-17
std      3.430972e-01
min     -1.149126e+00
25%     -1.787599e-01
50%     -3.741312e-02
75%      1.244052e-01
max      3.165676e+00
Name: cmi_gap, dtype: float64

MCC Gap Distribution:
count    3.266000e+03
mean    -9.790087e-18
std      2.422634e-01
min     -7.850369e-01
25%     -5.327803e-02
50%      6.730215e-03
75%      1.193143e-01
max      9.571624e-01
Name: mcc_gap, dtype: float64

CMI Z-Score Distribution:
count    3.176000e+03
mean    -1.924014e-16
std      9.733456e-01
min     -4.151866e+00
25%     -6.040665e-01
50%     -8.562270e-02
75%      4.305092e-01
max      6.404419e+00
Name: cmi_z_score, dtype: float64


## Identifying Potential Documentation Gaps

We use a multi-flag approach to identify hospitals with likely documentation gaps:

1. **below_peer_cmi**: CMI is 0.5 standard deviations below the peer group mean
2. **significant_gap**: CMI is 1.0 standard deviations below the peer group mean (more extreme)
3. **low_mcc_flag**: MCC ratio is 5+ percentage points below peer group mean
4. **documentation_risk**: Combined flag requiring both below-peer CMI AND low MCC ratio

Hospitals flagged for documentation risk warrant detailed chart review and coder training.

In [4]:
# Define flags for documentation gaps (only for hospitals with peer group stats)
has_peer_stats = df_benchmarked['n_hospitals'].notna()

df_benchmarked['below_peer_cmi'] = False
df_benchmarked.loc[has_peer_stats, 'below_peer_cmi'] = df_benchmarked.loc[has_peer_stats, 'cmi_z_score'] < -0.5

df_benchmarked['significant_gap'] = False
df_benchmarked.loc[has_peer_stats, 'significant_gap'] = df_benchmarked.loc[has_peer_stats, 'cmi_z_score'] < -1.0

df_benchmarked['low_mcc_flag'] = False
df_benchmarked.loc[has_peer_stats, 'low_mcc_flag'] = df_benchmarked.loc[has_peer_stats, 'mcc_gap'] < -0.05

df_benchmarked['documentation_risk'] = False
df_benchmarked.loc[has_peer_stats, 'documentation_risk'] = (
    df_benchmarked.loc[has_peer_stats, 'below_peer_cmi'] & 
    df_benchmarked.loc[has_peer_stats, 'low_mcc_flag']
)

# Print flag counts and percentages
print("Documentation Gap Flags (for hospitals with peer group stats):")
n_with_stats = has_peer_stats.sum()

for flag in ['below_peer_cmi', 'significant_gap', 'low_mcc_flag', 'documentation_risk']:
    n_flagged = df_benchmarked[flag].sum()
    pct_flagged = 100 * n_flagged / n_with_stats if n_with_stats > 0 else 0
    print(f"\n  {flag}:")
    print(f"    Count: {n_flagged}")
    print(f"    Percentage: {pct_flagged:.1f}%")

# Summary statistics
print(f"\nSummary:")
print(f"  Total hospitals with peer group stats: {n_with_stats}")
print(f"  Hospitals with documentation risk: {df_benchmarked['documentation_risk'].sum()}")

Documentation Gap Flags (for hospitals with peer group stats):

  below_peer_cmi:
    Count: 937
    Percentage: 28.7%

  significant_gap:
    Count: 345
    Percentage: 10.6%

  low_mcc_flag:
    Count: 843
    Percentage: 25.8%

  documentation_risk:
    Count: 288
    Percentage: 8.8%

Summary:
  Total hospitals with peer group stats: 3266
  Hospitals with documentation risk: 288


In [5]:
# Show top 20 hospitals with largest negative CMI gap
top_gaps = df_benchmarked[
    df_benchmarked['n_hospitals'].notna()
].nsmallest(20, 'cmi_gap')[[
    'ccn', 'hospital_name', 'state', 'beds', 'cmi', 'peer_cmi_mean', 'cmi_gap', 
    'mcc_ratio', 'peer_mcc_ratio_mean', 'cmi_z_score', 'documentation_risk'
]].copy()

print("Top 20 Hospitals with Largest Negative CMI Gap (Potential Documentation Opportunities):")
print()

# Format for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

display_cols = [
    'ccn', 'hospital_name', 'state', 'beds', 
    'cmi', 'peer_cmi_mean', 'cmi_gap', 'cmi_z_score',
    'mcc_ratio', 'peer_mcc_ratio_mean', 'documentation_risk'
]

print(top_gaps[display_cols].to_string(index=False))

# Also show stats on these high-opportunity hospitals
print(f"\n\nCharacteristics of Top 20 Gap Hospitals:")
print(f"  Average CMI gap: {top_gaps['cmi_gap'].mean():.4f}")
print(f"  Average CMI z-score: {top_gaps['cmi_z_score'].mean():.2f}")
print(f"  Percentage flagged for documentation risk: {100 * top_gaps['documentation_risk'].mean():.1f}%")
print(f"  Average bed size: {top_gaps['beds'].mean():.0f}")

Top 20 Hospitals with Largest Negative CMI Gap (Potential Documentation Opportunities):

   ccn                                   hospital_name state   beds    cmi  peer_cmi_mean   cmi_gap  cmi_z_score  mcc_ratio  peer_mcc_ratio_mean  documentation_risk
670314                ST MICHAELS MEDICAL HOSPITAL LLC    TX    7.0 0.8334       1.982526 -1.149126    -1.482259        0.0             0.042838               False
030154             EXCEPTIONAL COMMUNITY HOSPITAL YUMA    AZ    9.0 0.9116       1.982526 -1.070926    -1.381388        0.0             0.042838               False
030152       EXCEPTIONAL COMMUNITY HOSPITAL - MARICOPA    AZ    9.0 0.9159       1.982526 -1.066626    -1.375842        0.0             0.042838               False
670134                        ALTUS LUMBERTON HOSPITAL    TX    4.0 1.0225       2.080180 -1.057680    -1.471622        0.0             0.000000               False
460043                         OREM COMMUNITY HOSPITAL    UT   20.0 0.6543       1.706

In [6]:
# Save outputs
output_benchmarks = output_dir / 'hospital_with_benchmarks.csv'
output_summary = output_dir / 'peer_group_summary.csv'

# Save hospital-level data with benchmarks
df_benchmarked.to_csv(output_benchmarks, index=False)
print(f"Saved hospital benchmarks to: {output_benchmarks}")
print(f"  Shape: {df_benchmarked.shape}")

# Save peer group summary statistics
peer_stats_filtered.to_csv(output_summary, index=False)
print(f"\nSaved peer group summary to: {output_summary}")
print(f"  Shape: {peer_stats_filtered.shape}")

print(f"\nNB06 Complete!")
print(f"\nKey outputs:")
print(f"  1. hospital_with_benchmarks.csv - Hospital data with peer group metrics and gaps")
print(f"  2. peer_group_summary.csv - Aggregate statistics by peer group")

Saved hospital benchmarks to: ../../data/outputs/nb06_peer_benchmarks/hospital_with_benchmarks.csv
  Shape: (3280, 69)

Saved peer group summary to: ../../data/outputs/nb06_peer_benchmarks/peer_group_summary.csv
  Shape: (42, 11)

NB06 Complete!

Key outputs:
  1. hospital_with_benchmarks.csv - Hospital data with peer group metrics and gaps
  2. peer_group_summary.csv - Aggregate statistics by peer group
